In [1]:

# COLAB SETUP: Install and start Spark

!pip -q install pyspark

from pyspark.sql import SparkSession
from pyspark import SparkContext

spark = SparkSession.builder \
    .appName("Assignment1_PySpark_RDD") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark version:", spark.version)

Spark version: 4.0.2


In [2]:

# Q1: Create SparkSession + RDD from Python list

nums = [1, 2, 3, 4, 5, 6]
rdd_nums = sc.parallelize(nums)

# Display all elements
print("RDD elements:", rdd_nums.collect())

# Total number of elements
print("Total elements:", rdd_nums.count())

# Short explanation (you can paste into your report)
q1_explanation = """
Why RDDs are immutable:
- You cannot change an RDD in-place. Operations like map/filter create a NEW RDD.
- This makes distributed execution safer and simpler (no race conditions on shared data).

Why RDDs are fault-tolerant:
- Spark tracks lineage (the sequence of transformations used to build an RDD).
- If a partition is lost (machine failure), Spark recomputes it using the lineage.
"""
print(q1_explanation)

RDD elements: [1, 2, 3, 4, 5, 6]
Total elements: 6

Why RDDs are immutable:
- You cannot change an RDD in-place. Operations like map/filter create a NEW RDD.
- This makes distributed execution safer and simpler (no race conditions on shared data).

Why RDDs are fault-tolerant:
- Spark tracks lineage (the sequence of transformations used to build an RDD).
- If a partition is lost (machine failure), Spark recomputes it using the lineage.



In [3]:

# Q2: RDD from 1 to 10 -> map() squares, filter() evens

rdd_1_10 = sc.parallelize(list(range(1, 11)))

# map() square
squares = rdd_1_10.map(lambda x: x * x)

# filter() even numbers
evens = rdd_1_10.filter(lambda x: x % 2 == 0)

print("Original:", rdd_1_10.collect())
print("Squares:", squares.collect())
print("Evens:", evens.collect())

Original: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Squares: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens: [2, 4, 6, 8, 10]


In [4]:

# Q3: reduce() sum, count(), take(5)

from functools import reduce

sum_all = rdd_1_10.reduce(lambda a, b: a + b)
total_count = rdd_1_10.count()
first_five = rdd_1_10.take(5)

print("Sum using reduce():", sum_all)
print("Count using count():", total_count)
print("First five using take(5):", first_five)

Sum using reduce(): 55
Count using count(): 10
First five using take(5): [1, 2, 3, 4, 5]


In [5]:

# Q4: Load a text file into an RDD and analyze

# OPTION A (Recommended): Upload a file in Colab
from google.colab import files
uploaded = files.upload()  # choose a .txt file
txt_path = list(uploaded.keys())[0]
print("Uploaded:", txt_path)

# Load as RDD (each line is one element)
text_rdd = sc.textFile(txt_path)

# Count total lines
line_count = text_rdd.count()

# Count total words
words_rdd = text_rdd.flatMap(lambda line: line.split())
word_count = words_rdd.count()

# Find the longest word (basic cleanup: remove empty strings)
clean_words = words_rdd.map(lambda w: w.strip()).filter(lambda w: w != "")
longest_word = clean_words.reduce(lambda a, b: a if len(a) >= len(b) else b)

print("Total lines:", line_count)
print("Total words:", word_count)
print("Longest word:", longest_word, "(length:", len(longest_word), ")")

Saving sample.txt to sample (1).txt
Uploaded: sample (1).txt
Total lines: 10
Total words: 81
Longest word: general-purpose (length: 15 )


In [6]:

# Q5: Word Count using RDD (lowercase + top 10)

import re

def tokenize(line: str):
    # keep only words (letters/digits/underscore). Adjust if your teacher wants punctuation included.
    return re.findall(r"\w+", line.lower())

words = text_rdd.flatMap(tokenize)

word_freq = words.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)

top10 = word_freq.takeOrdered(10, key=lambda x: -x[1])  # descending by count
print("Top 10 most frequent words:")
for w, c in top10:
    print(w, "->", c)

Top 10 most frequent words:
spark -> 6
and -> 5
data -> 4
apache -> 2
level -> 2
rdd -> 2
distributed -> 2
is -> 2
a -> 2
in -> 2


In [8]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv"

!wget -O searchterms.csv "$url"
!ls -lh searchterms.csv

--2026-02-10 18:20:58--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 233457 (228K) [text/csv]
Saving to: ‘searchterms.csv’

searchterms.csv     100%[===================>] 227.99K  --.-KB/s    in 0.08s   

2026-02-10 18:20:59 (2.93 MB/s) - ‘searchterms.csv’ saved [233457/233457]

-rw-r--r-- 1 root root 228K Sep 29  2022 searchterms.csv


In [10]:
df = spark.read.csv("searchterms.csv", header=False, inferSchema=True) \
    .toDF("term", "col2", "col3", "col4")

df.show(5, truncate=False)
print("Total rows:", df.count())

+----+-----+----+--------------+
|term|col2 |col3|col4          |
+----+-----+----+--------------+
|day |month|year|searchterm    |
|12  |11   |2021|mobile 6 inch |
|12  |11   |2021|mobile latest |
|12  |11   |2021|tablet wifi   |
|12  |11   |2021|laptop 14 inch|
+----+-----+----+--------------+
only showing top 5 rows
Total rows: 10001


In [11]:
threshold = 100

term_counts = df.groupBy("term").count()
filtered = term_counts.filter(F.col("count") > threshold)

filtered.orderBy(F.desc("count")).show(10, truncate=False)
print("Unique terms with count >", threshold, ":", filtered.count())

+----+-----+
|term|count|
+----+-----+
|18  |465  |
|15  |465  |
|22  |465  |
|19  |465  |
|23  |465  |
|14  |465  |
|13  |464  |
|16  |464  |
|17  |464  |
|24  |464  |
+----+-----+
only showing top 10 rows
Unique terms with count > 100 : 30


In [12]:
# 2) Average length of search terms
avg_len = df.select(F.avg(F.length("term")).alias("avg_length")).collect()[0]["avg_length"]
print("Average search term length:", avg_len)

Average search term length: 1.7911208879112088


In [13]:
# 3) Count search terms grouped by first letter
first_letter_counts = df.select(
    F.lower(F.substring("term", 1, 1)).alias("first_letter")
).groupBy("first_letter").count().orderBy("first_letter")

first_letter_counts.show(30, truncate=False)

+------------+-----+
|first_letter|count|
+------------+-----+
|1           |4217 |
|2           |3924 |
|3           |465  |
|4           |233  |
|5           |232  |
|6           |232  |
|7           |232  |
|8           |233  |
|9           |232  |
|d           |1    |
+------------+-----+



In [14]:
# 4) How many times was the term `gaming laptop` searched?
gaming_laptop_count = df.select(F.lower(F.col("term")).alias("t")) \
    .filter(F.col("t") == "gaming laptop") \
    .count()

print("`gaming laptop` searched:", gaming_laptop_count, "times")

`gaming laptop` searched: 0 times
